# Data Profiling Notebook - Spotify x Grammy Awards

**Course:** ETL / Data Engineering - Workshop 2
**Purpose:** establish the *evidence base* for the data-quality rule catalogue
(`src/validation.py`), the transformation contract (`src/transform.py`) and the
analytical requirements (`docs/analytical_requirements.md`) by profiling the two
raw sources before any validation or transformation runs.

| Item | Value |
| --- | --- |
| Source A | `data/raw/spotify_dataset.csv` (Spotify tracks, Kaggle) |
| Source B | `data/raw/the_grammy_awards.csv` (Grammy Awards 1958-2019) |
| Notebook type | Batch profiling / contract discovery (read-only) |
| Kernel | Python 3 + pandas |
| Output of this notebook | Inline evidence + `docs/evidence/profiling_summary.json` |

**Profiling dimensions covered:** structure, completeness, uniqueness, categorical
distribution, numerical range/validity, temporal coverage, cross-source
consistency (the integration dimension that drives the Grammy <-> Spotify match
rate and the `dq_cross_*` rule group).

> Every number below is produced by executing this notebook against the raw CSVs -
> it is reproducible evidence, not documentation written by hand.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 120)

PROJECT = Path.cwd().resolve()
if not (PROJECT / "src").is_dir():          # allow running from notebooks/
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

from src.config import GRAMMY_REQUIRED_COLUMNS, SPOTIFY_REQUIRED_COLUMNS, read_csv
from src.transform import _split_artists, normalize_artist_name

DATA = PROJECT / "data" / "raw"
EVIDENCE = PROJECT / "docs" / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)

# Same NA policy as src/config.read_csv: the Spotify file contains the literal
# token "N/A" as a real artist name, so pandas' default NA parsing would
# corrupt it.
spotify = read_csv(DATA / "spotify_dataset.csv")
grammys = read_csv(DATA / "the_grammy_awards.csv", dtype={"winner": "string"})

print("spotify :", spotify.shape)
print("grammys :", grammys.shape)
print("production normalization + split helpers imported from src.transform")

spotify : (114000, 21)
grammys : (4810, 10)
production normalization + split helpers imported from src.transform


## 1. Structure profiling

Columns, dtypes, memory footprint and the declared contract of each source.

In [ ]:
structure = pd.DataFrame({
    "dtype": spotify.dtypes.astype(str),
    "non_null": spotify.notna().sum(),
    "nulls": spotify.isna().sum(),
    "unique": spotify.nunique(dropna=True),
})
structure["null_pct"] = (structure["nulls"] / len(spotify) * 100).round(4)
structure

,dtype,non_null,nulls,unique,null_pct
Unnamed: 0,int64,114000,0,114000,0.0000
track_id,object,114000,0,89741,0.0000
artists,object,113999,1,31437,0.0009
album_name,object,113999,1,46589,0.0009
track_name,object,113999,1,73608,0.0009
popularity,int64,114000,0,101,0.0000
duration_ms,int64,114000,0,50697,0.0000
explicit,bool,114000,0,2,0.0000
danceability,float64,114000,0,1174,0.0000
energy,float64,114000,0,2083,0.0000


In [ ]:
# Extract contract: which raw columns survive into the pipeline.
contract = pd.DataFrame(
    {"in_extract_contract": [col in SPOTIFY_REQUIRED_COLUMNS for col in spotify.columns]},
    index=spotify.columns,
)
print("raw spotify columns :", len(spotify.columns))
print("contract columns    :", len(SPOTIFY_REQUIRED_COLUMNS))
print("dropped by extract  :",
      [c for c in spotify.columns if c not in SPOTIFY_REQUIRED_COLUMNS])
print()
print("raw grammy columns  :", len(grammys.columns))
print("grammy contract     :", len(GRAMMY_REQUIRED_COLUMNS))
print("dropped by extract  :",
      [c for c in grammys.columns if c not in GRAMMY_REQUIRED_COLUMNS])
display(contract[~contract["in_extract_contract"]])

raw spotify columns : 21
contract columns    : 16
dropped by extract  : ['Unnamed: 0', 'key', 'mode', 'instrumentalness', 'time_signature']

raw grammy columns  : 10
grammy contract     : 9
dropped by extract  : ['img']


,in_extract_contract
Unnamed: 0,False
key,False
mode,False
instrumentalness,False
time_signature,False


In [ ]:
structure_g = pd.DataFrame({
    "dtype": grammys.dtypes.astype(str),
    "non_null": grammys.notna().sum(),
    "nulls": grammys.isna().sum(),
    "unique": grammys.nunique(dropna=True),
})
structure_g["null_pct"] = (structure_g["nulls"] / len(grammys) * 100).round(4)
structure_g

,dtype,non_null,nulls,unique,null_pct
year,int64,4810,0,62,0.0000
title,object,4810,0,62,0.0000
published_at,object,4810,0,4,0.0000
updated_at,object,4810,0,10,0.0000
category,object,4810,0,638,0.0000
nominee,object,4804,6,4131,0.1247
artist,object,2970,1840,1658,38.2536
workers,object,2620,2190,2366,45.5301
img,object,3443,1367,1463,28.4200
winner,string,4810,0,1,0.0000


## 2. Completeness profiling

Blank / null detection per column. These counts are the direct evidence behind
**DQ-S1..S6** (Spotify criticality) and **DQ-G1..G5** (Grammys), including the
documented *mostly* tolerances (e.g. 99.9% for `artists`).

In [ ]:
def completeness(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "nulls": df.isna().sum(),
        "blank_strings": {
            col: int((df[col].astype("string").str.strip() == "").sum())
            for col in df.columns
        },
    })
    out["missing_total"] = out["nulls"] + out["blank_strings"]
    out["missing_pct"] = (out["missing_total"] / len(df) * 100).round(4)
    out.insert(0, "source", name)
    return out

completeness(spotify, "spotify")

,source,nulls,blank_strings,missing_total,missing_pct
Unnamed: 0,spotify,0,0,0,0.0000
track_id,spotify,0,0,0,0.0000
artists,spotify,1,0,1,0.0009
album_name,spotify,1,0,1,0.0009
track_name,spotify,1,0,1,0.0009
popularity,spotify,0,0,0,0.0000
duration_ms,spotify,0,0,0,0.0000
explicit,spotify,0,0,0,0.0000
danceability,spotify,0,0,0,0.0000
energy,spotify,0,0,0,0.0000


In [ ]:
completeness(grammys, "grammys")

,source,nulls,blank_strings,missing_total,missing_pct
year,grammys,0,0,0,0.0000
title,grammys,0,0,0,0.0000
published_at,grammys,0,0,0,0.0000
updated_at,grammys,0,0,0,0.0000
category,grammys,0,0,0,0.0000
nominee,grammys,6,0,6,0.1247
artist,grammys,1840,0,1840,38.2536
workers,grammys,2190,0,2190,45.5301
img,grammys,1367,0,1367,28.4200
winner,grammys,0,0,0,0.0000


**Grammy artist-cardinality evidence** - `artist` is the column that drives the
cross-source integration key, so its missingness is an integration risk
(AR1-AR4: awards without an artist credit cannot be matched to Spotify).

In [ ]:
g_missing_artist = grammys["artist"].isna().sum()
print(f"rows without artist credit : {g_missing_artist} / {len(grammys)} "
      f"({g_missing_artist / len(grammys) * 100:.2f}%)")
print("rows with a non-empty artist credit:",
      int(grammys["artist"].notna().sum()))

rows without artist credit : 1840 / 4810 (38.25%)
rows with a non-empty artist credit: 2970


## 3. Uniqueness / grain profiling

Candidate keys, duplicate counts and the grain adopted by the dimensional model.

In [ ]:
dup_track_id = int(spotify["track_id"].duplicated().sum())
dup_grain = int(spotify.duplicated(subset=["track_id", "track_genre"]).sum())
dup_full = int(spotify.duplicated().sum())

print(f"duplicate track_id                    : {dup_track_id}")
print(f"duplicate (track_id, track_genre) grain: {dup_grain}")
print(f"fully duplicated rows                  : {dup_full}")
print(f"distinct track_id                      : {spotify['track_id'].nunique()}")
print(f"-> grain adopted: (track_id, track_genre) = {len(spotify) - dup_grain} rows")

duplicate track_id                    : 24259
duplicate (track_id, track_genre) grain: 450
fully duplicated rows                  : 0
distinct track_id                      : 89741
-> grain adopted: (track_id, track_genre) = 113550 rows


In [ ]:
# Grammy candidate business key: year | category | nominee | artist.
# NOTE: concatenating a nullable string column with a NaN component propagates
# pd.NA to the whole key, which would falsely report every null-artist row as a
# duplicate. fillna("") first - exactly as src.transform builds award_bk.
award_bk = (
    grammys["year"].astype("int64").astype(str)
    + "|"
    + grammys["category"].fillna("").astype(str)
    + "|"
    + grammys["nominee"].fillna("").astype(str)
    + "|"
    + grammys["artist"].fillna("").astype(str)
)
dup_award = int(award_bk.duplicated().sum())
naive_bk = (
    grammys["year"].astype("string") + "|"
    + grammys["category"].astype("string") + "|"
    + grammys["nominee"].astype("string") + "|"
    + grammys["artist"].astype("string")
)
print(f"duplicate award business key (null-safe) : {dup_award}")
print(f"duplicate award business key (naive NA)  : {int(naive_bk.duplicated().sum())}"
      "  <- artifact of pd.NA propagation, not a data defect")
print("-> award_bk is unique; it is a safe business key for fact_grammy_award")

duplicate award business key (null-safe) : 0
duplicate award business key (naive NA)  : 1839  <- artifact of pd.NA propagation, not a data defect
-> award_bk is unique; it is a safe business key for fact_grammy_award


## 4. Categorical profiling

In [ ]:
genre_counts = spotify["track_genre"].value_counts()
print("distinct track_genre values:", genre_counts.size)
print("rows per genre: min=%d max=%d mean=%.1f"
      % (genre_counts.min(), genre_counts.max(), genre_counts.mean()))
display(pd.concat([genre_counts.head(8), genre_counts.tail(8)]))

print()
print("explicit flag:", spotify["explicit"].value_counts().to_dict())
print("time_signature oddities:",
      int((spotify["time_signature"] == 0).sum()), "zero values,",
      int((~spotify["time_signature"].isin([3, 4, 5, 6, 7])).sum()), "outside 3-7")

distinct track_genre values: 114
rows per genre: min=1000 max=1000 mean=1000.0


track_genre
acoustic       1000
afrobeat       1000
alt-rock       1000
alternative    1000
ambient        1000
anime          1000
black-metal    1000
bluegrass      1000
swedish        1000
synth-pop      1000
tango          1000
techno         1000
trance         1000
trip-hop       1000
turkish        1000
world-music    1000
Name: count, dtype: int64


explicit flag: {False: 104253, True: 9747}
time_signature oddities: 163 zero values, 1136 outside 3-7


In [ ]:
cat = pd.DataFrame({
    "distinct": grammys.nunique(dropna=True),
    "top_value": [grammys[c].dropna().astype(str).value_counts().index[0]
                  if grammys[c].notna().any() else None for c in grammys.columns],
    "top_value_rows": [int(grammys[c].value_counts().iloc[0])
                       if grammys[c].notna().any() else 0 for c in grammys.columns],
})
display(cat)

print("distinct award categories :", grammys["category"].nunique())
print("winner flag values        :", grammys["winner"].value_counts().to_dict())
print("top categories:")
display(grammys["category"].value_counts().head(10))

,distinct,top_value,top_value_rows
year,62,2019,433
title,62,62nd Annual GRAMMY Awards (2019),433
published_at,4,2017-11-28T00:03:45-08:00,4205
updated_at,10,2019-09-10T01:08:19-07:00,778
category,638,Song Of The Year,70
nominee,4131,Bridge Over Troubled Water,7
artist,1658,(Various Artists),66
workers,2366,"John Williams, composer (John Williams)",20
img,1463,https://www.grammy.com/sites/com/files/styles/...,26
winner,1,True,4810


distinct award categories : 638
winner flag values        : {'True': 4810}
top categories:


category
Song Of The Year                  70
Record Of The Year                69
Album Of The Year                 66
Best Opera Recording              64
Best Album Notes                  63
Best Instrumental Composition     55
Best Country Song                 55
Best New Artist                   51
Best Historical Album             44
Best Chamber Music Performance    43
Name: count, dtype: int64

## 5. Numerical profiling (range / validity)

Evidence for the gated range rules:

| Rule | Column(s) | Expectation | Severity |
| --- | --- | --- | --- |
| DQ-S3 | `popularity` | 0 <= value <= 100 | critical |
| DQ-S4 | `duration_ms` | 1 <= value <= 600000, mostly 0.99 | warning |
| DQ-S5 | `danceability`, `energy`, `valence` | 0 <= value <= 1 | critical |

Columns such as `loudness`, `tempo` and `time_signature` are profiled below for
completeness of the analysis, but they are **outside the extract contract**
(see the cell above) and therefore are reported, never gated - they also
illustrate rule **T5**: no value repair, out-of-range source data is reported
rather than silently corrected.

In [ ]:
numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy", "valence",
    "acousticness", "speechiness", "liveness", "loudness", "tempo",
    "explicit", "key", "mode", "time_signature",
]
desc = spotify[numeric_cols].describe().T[["count", "min", "max", "mean", "50%", "std"]]
desc

,count,min,max,mean,50%,std
popularity,114000.0,0.000,100.000,33.238535,35.0000,22.305078
duration_ms,114000.0,0.000,5237295.000,228029.153114,212906.0000,107297.712645
danceability,114000.0,0.000,0.985,0.566800,0.5800,0.173542
energy,114000.0,0.000,1.000,0.641383,0.6850,0.251529
valence,114000.0,0.000,0.995,0.474068,0.4640,0.259261
acousticness,114000.0,0.000,0.996,0.314910,0.1690,0.332523
speechiness,114000.0,0.000,0.965,0.084652,0.0489,0.105732
liveness,114000.0,0.000,1.000,0.213553,0.1320,0.190378
loudness,114000.0,-49.531,4.532,-8.258960,-7.0040,5.029337
tempo,114000.0,0.000,243.372,122.147837,122.0170,29.978197


In [ ]:
checks = {
    "popularity < 0": int((spotify["popularity"] < 0).sum()),
    "popularity > 100": int((spotify["popularity"] > 100).sum()),
    "duration_ms <= 0": int((spotify["duration_ms"] <= 0).sum()),
    "duration_ms > 600000 (>10 min)": int((spotify["duration_ms"] > 600_000).sum()),
    "duration_ms > 3600000 (>1 h)": int((spotify["duration_ms"] > 3_600_000).sum()),
    "energy outside [0,1]": int((~spotify["energy"].between(0, 1)).sum()),
    "loudness > 0 dB": int((spotify["loudness"] > 0).sum()),
    "loudness < -60 dB": int((spotify["loudness"] < -60).sum()),
    "tempo <= 0": int((spotify["tempo"] <= 0).sum()),
    "time_signature == 0": int((spotify["time_signature"] == 0).sum()),
}
pd.Series(checks, name="rows_failing")

popularity < 0                      0
popularity > 100                    0
duration_ms <= 0                    1
duration_ms > 600000 (>10 min)    603
duration_ms > 3600000 (>1 h)       16
energy outside [0,1]                0
loudness > 0 dB                    90
loudness < -60 dB                   0
tempo <= 0                        157
time_signature == 0               163
Name: rows_failing, dtype: int64

Grammy-side numerics (year coverage, the only truly numeric dimension in that source).

In [ ]:
print("year min / max:", grammys["year"].min(), "/", grammys["year"].max())
print("distinct years :", grammys["year"].nunique())
years = grammys["year"].value_counts().sort_index()
missing_years = sorted(set(range(grammys["year"].min(), grammys["year"].max() + 1)) - set(years.index))
print("years with zero awards:", missing_years or "none")
display(years.head(3))
display(years.tail(3))
print("published_at parseable:",
      int(pd.to_datetime(grammys["published_at"], errors="coerce").notna().sum()), "/", len(grammys))

year min / max: 1958 / 2019
distinct years : 62
years with zero awards: none


year
1958    28
1959    35
1960    39
Name: count, dtype: int64

year
2017     86
2018     86
2019    433
Name: count, dtype: int64

published_at parseable: 4810 / 4810


/tmp/ipykernel_91717/264380556.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  int(pd.to_datetime(grammys["published_at"], errors="coerce").notna().sum()), "/", len(grammys))


## 6. Cross-source consistency (integration profiling)

The integration key is `artist_key = normalize_artist_name(artist credit)`.
This notebook imports the **production implementation** from `src.transform`
instead of re-implementing it, so the numbers here are by construction the same
ones the transform stage reports:

- NFKD accent folding, lowercase, whitespace collapse
- surrounding quotes/periods stripped
- `"..." (Various Artists) ..."` canonicalised to `various artists`
- Spotify credits split on `;` (one track can have 1..n artists)
- Grammy side: the whole `artist` string is one key (nullable, no split)

Cardinality contract: Spotify track row -> 1..n artists; Grammy award row ->
0..1 artist_key; unmatched rows are retained with `is_matched_spotify = 0`
(never dropped) and measured instead.

In [ ]:
# Production integration contract (src.transform.normalize_artist_name and
# _split_artists are imported so this notebook cannot drift from the pipeline).
spotify_keys = set()
for value in spotify["artists"].dropna():
    for _display, key in _split_artists(value):
        spotify_keys.add(key)

grammy_keys = {normalize_artist_name(v) for v in grammys["artist"].dropna()}
grammy_keys.discard(None)

grammy_artist_key = grammys["artist"].map(normalize_artist_name)
matched_mask = grammy_artist_key.isin(spotify_keys)
award_rows_with_artist = int(grammys["artist"].notna().sum())
award_rows_matched = int(matched_mask.sum())
matched = spotify_keys & grammy_keys

print("distinct artist keys - spotify :", len(spotify_keys))
print("distinct artist keys - grammys :", len(grammy_keys))
print("matched artist keys            :", len(matched))
print("award rows with artist credit  :", award_rows_with_artist)
print("award rows matched to spotify  :", award_rows_matched)
print("match rate (rows/w.artist rows):",
      round(award_rows_matched / award_rows_with_artist * 100, 4), "%")
print("match rate (rows/all rows)     :",
      round(award_rows_matched / len(grammys) * 100, 4), "%")
print("unmatched award rows           :", len(grammys) - award_rows_matched)
print("  - of which no artist credit  :", len(grammys) - award_rows_with_artist)
print("  - of which no Spotify overlap:",
      award_rows_with_artist - award_rows_matched)
display(sorted(matched)[:15])

distinct artist keys - spotify : 29789
distinct artist keys - grammys : 1636
matched artist keys            : 531
award rows with artist credit  : 2970
award rows matched to spotify  : 1395
match rate (rows/w.artist rows): 46.9697 %
match rate (rows/all rows)     : 29.0021 %
unmatched award rows           : 3415
  - of which no artist credit  : 1840
  - of which no Spotify overlap: 1575


['21 savage',
 'a flock of seagulls',
 'ac/dc',
 'adele',
 'aerosmith',
 'aimee mann',
 'al green',
 'al hirt',
 'alabama shakes',
 'alanis morissette',
 'alejandro sanz',
 'alicia keys',
 'alison krauss',
 'alison krauss & union station',
 'amy winehouse']

In [ ]:
# Ambiguity evidence: one normalized key produced by several raw display names
# (same measurement as `normalized_display_name_conflicts` in transform metrics).
display_rows = pd.concat(
    [
        pd.DataFrame({
            "artist_key": spotify["artists"].dropna().map(
                lambda v: [k for _d, k in _split_artists(v)]
            ).explode(),
            "display_name": spotify["artists"].dropna().map(
                lambda v: [d for d, _k in _split_artists(v)]
            ).explode(),
        }),
        grammys.dropna(subset=["artist"])[["artist"]].rename(
            columns={"artist": "display_name"}
        ).assign(
            artist_key=grammys.dropna(subset=["artist"])["artist"].map(
                normalize_artist_name
            )
        ),
    ],
    ignore_index=True,
).dropna()

conflicts = (
    display_rows.groupby("artist_key")["display_name"].nunique().loc[lambda s: s > 1]
)
print("normalized keys with >1 raw display name:", len(conflicts))
for key in conflicts.index[:8]:
    raws = sorted(set(display_rows.loc[display_rows["artist_key"] == key, "display_name"]))
    print(f"  {key!r} <- {raws}")

normalized keys with >1 raw display name: 90
  '7 minutoz' <- ['7 MinutoZ', '7 Minutoz']
  'alaska' <- ['ALASKA', 'Alaska']
  'andre previn' <- ['Andre Previn', 'André Previn']
  'andres' <- ['Andres', 'Andrés']
  'aurora' <- ['AURORA', 'Aurora']
  'bijou' <- ['BIJOU', 'Bijou']
  'brave' <- ['BRAVE', 'Brave']
  'brianna' <- ['BRIANNA', 'Brianna']


## 7. Profiling findings -> data-quality rules

In [ ]:
findings = pd.DataFrame([
    ["structure",
     f"raw Spotify CSV has {len(spotify.columns)} columns; extract keeps the {len(SPOTIFY_REQUIRED_COLUMNS)} contract columns",
     "extract.py column contract + DQ-S1..S6 on contract columns", "info"],
    ["completeness",
     f"artists null in {int(spotify['artists'].isna().sum())} row(s)",
     "DQ-S2 critical (mostly=0.999), T1 drops the row", "warning"],
    ["completeness",
     "no nulls in track_id / track_name / track_genre / audio features",
     "DQ-S1 / DQ-S6 critical", "critical"],
    ["completeness",
     f"grammy artist null in {g_missing_artist} rows ({g_missing_artist/len(grammys)*100:.2f}%)",
     "DQ-G5 warning (mostly=0.60) + documented AR1-AR4 limitation", "warning"],
    ["uniqueness",
     f"{dup_grain} duplicated (track_id, track_genre) rows",
     "T2 drop -> DQ-P10 critical (prepared duplicate grain = 0)", "critical"],
    ["uniqueness",
     f"{dup_award} duplicated award business keys (null-safe award_bk)",
     "award_bk is a safe business key for fact_grammy_award", "info"],
    ["validity",
     "popularity outside 0-100 in the injected bad file",
     "DQ-S3 critical range rule (Test B blocks the run)", "critical"],
    ["validity",
     f"{int((spotify['duration_ms'] > 600000).sum())} rows > 10 min, {int((spotify['duration_ms'] <= 0).sum())} row = 0 ms",
     "DQ-S4 warning band (max_value=600000, no silent repair - T5)", "warning"],
    ["validity",
     f"{int((spotify['time_signature'] == 0).sum())} rows with time_signature 0, {int((spotify['tempo'] <= 0).sum())} rows with tempo <= 0",
     "outside extract contract -> documented, not gated (T5 / scope note)", "warning"],
    ["categorical",
     f"{genre_counts.size} genres, {genre_counts.min()}..{genre_counts.max()} rows each",
     "dim_genre grain + AR2 (genre split)", "info"],
    ["categorical",
     "winner == True for all 4,810 Grammy rows",
     "documented source limitation: dataset contains winners only", "info"],
    ["temporal",
     f"grammy years {grammys['year'].min()}-{grammys['year'].max()}, {grammys['year'].nunique()} distinct, no missing years",
     "dim_year grain + AR3 (decade evolution)", "info"],
    ["integration",
     f"match rate {award_rows_matched/award_rows_with_artist*100:.4f}% of artist-credited rows "
     f"({award_rows_matched/len(grammys)*100:.4f}% of all rows)",
     "integration baseline for AR1-AR4, KPI-0, DQ-P11 warning threshold", "critical"],
    ["integration",
     f"{len(conflicts)} normalized keys with multiple display names",
     "dim_artist surrogate key absorbs the merge; metric reported", "warning"],
    ["integration",
     f"{len(grammys) - award_rows_matched} award rows unmatched "
     f"({len(grammys) - award_rows_with_artist} without artist + "
     f"{award_rows_with_artist - award_rows_matched} without overlap)",
     "AR1-AR4 documented limitation: name-based matching, no artist IDs", "warning"],
], columns=["dimension", "evidence", "rule / decision", "risk"])

display(findings)

summary = {
    "sources": {
        "spotify_rows": int(len(spotify)),
        "spotify_raw_columns": int(spotify.shape[1]),
        "spotify_contract_columns": len(SPOTIFY_REQUIRED_COLUMNS),
        "spotify_dropped_columns": [c for c in spotify.columns if c not in SPOTIFY_REQUIRED_COLUMNS],
        "grammy_rows": int(len(grammys)),
        "grammy_raw_columns": int(grammys.shape[1]),
        "grammy_contract_columns": len(GRAMMY_REQUIRED_COLUMNS),
    },
    "grain": {
        "spotify_duplicate_track_id": dup_track_id,
        "spotify_duplicate_track_genre_grain": dup_grain,
        "spotify_fully_duplicated_rows": dup_full,
        "grammy_duplicate_award_bk": dup_award,
        "grammy_duplicate_award_bk_naive_na_artifact": int(naive_bk.duplicated().sum()),
    },
    "completeness": {
        "spotify_artists_null_rows": int(spotify["artists"].isna().sum()),
        "grammy_artist_null_rows": int(g_missing_artist),
        "grammy_artist_null_pct": round(g_missing_artist / len(grammys) * 100, 4),
    },
    "validity_flags": checks,
    "categoricals": {
        "distinct_genres": int(genre_counts.size),
        "distinct_grammy_categories": int(grammys["category"].nunique()),
        "grammy_winner_values": {str(k): int(v) for k, v in grammys["winner"].value_counts().items()},
    },
    "temporal": {
        "grammy_year_min": int(grammys["year"].min()),
        "grammy_year_max": int(grammys["year"].max()),
        "grammy_distinct_years": int(grammys["year"].nunique()),
    },
    "integration": {
        "normalization_source": "src.transform.normalize_artist_name (production code)",
        "spotify_artist_keys": len(spotify_keys),
        "grammy_artist_keys": len(grammy_keys),
        "matched_artist_keys": len(matched),
        "award_rows_with_artist": award_rows_with_artist,
        "award_rows_matched": award_rows_matched,
        "grammy_match_rate_pct_of_artist_rows": round(award_rows_matched / award_rows_with_artist * 100, 4),
        "grammy_match_rate_pct_of_all_rows": round(award_rows_matched / len(grammys) * 100, 4),
        "normalized_display_name_conflicts": int(len(conflicts)),
    },
}

out_path = EVIDENCE / "profiling_summary.json"
out_path.write_text(json.dumps(summary, indent=2))
print("wrote", out_path)

,dimension,evidence,rule / decision,risk
0,structure,raw Spotify CSV has 21 columns; extract keeps ...,extract.py column contract + DQ-S1..S6 on cont...,info
1,completeness,artists null in 1 row(s),"DQ-S2 critical (mostly=0.999), T1 drops the row",warning
2,completeness,no nulls in track_id / track_name / track_genr...,DQ-S1 / DQ-S6 critical,critical
3,completeness,grammy artist null in 1840 rows (38.25%),DQ-G5 warning (mostly=0.60) + documented AR1-A...,warning
4,uniqueness,"450 duplicated (track_id, track_genre) rows",T2 drop -> DQ-P10 critical (prepared duplicate...,critical
5,uniqueness,0 duplicated award business keys (null-safe aw...,award_bk is a safe business key for fact_gramm...,info
6,validity,popularity outside 0-100 in the injected bad file,DQ-S3 critical range rule (Test B blocks the run),critical
7,validity,"603 rows > 10 min, 1 row = 0 ms","DQ-S4 warning band (max_value=600000, no silen...",warning
8,validity,"163 rows with time_signature 0, 157 rows with ...","outside extract contract -> documented, not ga...",warning
9,categorical,"114 genres, 1000..1000 rows each",dim_genre grain + AR2 (genre split),info


wrote /home/drz/Documents/semesterV/etl/Activities/Workshop/2/docs/evidence/profiling_summary.json
